# Optical Flow

### What Is Optical Flow?

YOLO asks:
- what object is in this region?

Optical flow asks:
- where did this pixel move between frame 1 and frame 2?

No classes, no bounding boxes, no labels:
- just raw pixel movement
- no neural network needed
- works on any moving object
- tells us direction and speed of movement

Example — a dot moving right:

```
Frame 1:        Frame 2:        Optical Flow:
. . . . .       . . . . .       . . . . .
. . o . .   →   . . . o .   →   . . → → .
. . . . .       . . . . .       . . . . .
```

### Dense vs Sparse

**Dense optical flow:**
- calculates movement for every single pixel
- full motion picture of entire scene
- computationally heavy

**Sparse optical flow:**
- tracks only specific interesting points
- pick ~100 good corners, track only those
- much faster

We use sparse — a few key points tell us everything about movement without processing every pixel.

### Good Features To Track

Not all pixels are equally trackable.

Bad point — flat blue sky:
- every surrounding pixel looks identical
- if it moves 5 pixels → still looks exactly the same
- impossible to confirm it moved

Good point — chess board corner:
- distinct color change in 2 directions
- if it moves 5 pixels → clearly in a different place
- easy to confirm movement

This uniqueness is called **cornerness**.

`cv2.goodFeaturesToTrack()` finds corners with a quality threshold:
- minimum quality score
- minimum distance between points (no clustering)
- strong enough contrast to survive across frames

A regular corner detector finds any corner.
`goodFeaturesToTrack` only returns corners worth tracking.

### Lucas-Kanade Algorithm

The most widely used sparse optical flow method:
- takes previous frame, current frame, and points to track
- returns new positions of those points
- returns a status flag per point (tracked or lost)

Pipeline:

```
Frame 1
↓
goodFeaturesToTrack() → find ~100 good corners
↓
Frame 2
↓
calcOpticalFlowPyrLK() → where did those corners move?
↓
Update trails → draw path of each point
```

#### Why PyrLK?

`Pyr` = pyramid:
- runs on multiple downscaled versions of the frame
- coarse level catches large movements
- fine level refines small movements
- handles both fast and slow motion reliably

### Motion Trails — How They Work

Instead of drawing one arrow per frame, we store each point's position history:

```
trails = {
    0: [(245,130), (248,133), (251,137)],  ← point 0 history
    1: [(100,200), (102,198), (104,196)],  ← point 1 history
}
```

Then draw lines connecting all positions:

```
point moves:  A → B → C → D
trail draws:  A─B─C─D
```

#### How Points Keep Their Identity

Lucas-Kanade output is index-matched:
- `prev_points[0]` moved to `next_points[0]`
- `prev_points[1]` moved to `next_points[1]`

So the index IS the ID — no distance matching needed:
- `status[i] == 1` → point tracked → append to `trails[i]`
- `status[i] == 0` → point lost → delete `trails[i]`

Trails reset when we refresh points — acceptable since we refresh infrequently.

### Setup

In [19]:
import cv2
import numpy as np
from collections import deque

#### `deque`
- a list with a maximum length
- when full, oldest item is automatically removed
- perfect for trails — we only want the last N positions

```python
trail = deque(maxlen=20)
trail.append((1,1))
trail.append((2,2))
# after 20 items, oldest auto-removed
```

### Constants

In [ ]:
# goodFeaturesToTrack parameters
MAX_CORNERS  = 100
QUALITY      = 0.3
MIN_DISTANCE = 7

# Lucas-Kanade parameters
LK_PARAMS = dict(
    winSize  = (15, 15),
    maxLevel = 2,
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
)

TRAIL_LENGTH   = 20    # how many positions to remember per point
MIN_MOVEMENT   = 2     # minimum pixels moved to draw trail
REFRESH_EVERY  = 30    # refresh tracking points every N frames

#### `TRAIL_LENGTH`
- how many past positions to store per point
- longer = more visible trail, more memory

#### `REFRESH_EVERY`
- how often to find new tracking points
- every frame → trails reset too fast, invisible
- every 30 frames → trails build up visibly

### Webcam Loop

In [21]:
cap = cv2.VideoCapture(0)

ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

prev_points = cv2.goodFeaturesToTrack(
    prev_gray,
    maxCorners=MAX_CORNERS,
    qualityLevel=QUALITY,
    minDistance=MIN_DISTANCE
)

# trails[i] = deque of (x, y) positions for point i
trails = {}
frame_count = 0

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Track
    if prev_points is not None and len(prev_points) > 0:

        next_points, status, _ = cv2.calcOpticalFlowPyrLK(
            prev_gray, gray, prev_points, None, **LK_PARAMS
        )

        new_trails = {}

        for i, (prev_pt, next_pt, tracked) in enumerate(zip(prev_points, next_points, status)):

            if not tracked[0]:
                continue

            x1, y1 = map(int, prev_pt.ravel())
            x2, y2 = map(int, next_pt.ravel())

            movement = np.sqrt((x2 - x1)**2 + (y2 - y1)**2)

            if movement < MIN_MOVEMENT:
                continue

            # Get existing trail or create new one
            trail = trails.get(i, deque(maxlen=TRAIL_LENGTH))
            trail.append((x2, y2))
            new_trails[i] = trail

        trails = new_trails

        # Draw trails
        for trail in trails.values():
            pts = list(trail)
            for j in range(1, len(pts)):
                cv2.line(frame, pts[j-1], pts[j], (0, 255, 0), 2)
            cv2.circle(frame, pts[-1], 3, (0, 0, 255), -1)

    # Refresh points
    if frame_count % REFRESH_EVERY == 0 or prev_points is None or len(prev_points) < 10:
        prev_points = cv2.goodFeaturesToTrack(
            gray,
            maxCorners=MAX_CORNERS,
            qualityLevel=QUALITY,
            minDistance=MIN_DISTANCE
        )
        trails = {}
    else:
        prev_points = next_points[status == 1].reshape(-1, 1, 2)

    prev_gray = gray

    cv2.imshow("Optical Flow", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

#### `cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)`
- optical flow works on grayscale
- color adds no useful information for movement detection
- reduces computation

#### `cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_points, None, **LK_PARAMS)`
- `prev_gray` — previous frame
- `gray` — current frame
- `prev_points` — points to track from previous frame
- `None` — output array, we let OpenCV allocate it
- returns `next_points`, `status`, `error`

#### `status == 1`
- status is 1 if point was tracked successfully
- status is 0 if point was lost (moved out of frame, occluded)
- we only keep successfully tracked points

#### `.ravel()`
- points are stored as shape `(1, 2)` by OpenCV
- `.ravel()` flattens to `(2,)` so we can unpack as `x, y`

#### `cv2.arrowedLine()`
- draws an arrow from previous position to new position
- direction shows movement direction
- length shows how far the point moved

#### `cv2.circle()`
- marks the original point position in red
- makes it easier to see where tracking started

#### `for i, (prev_pt, next_pt, tracked) in enumerate(zip(...))`
- iterates all three arrays together
- `i` is the point index — used as trail ID

#### `trails.get(i, deque(maxlen=TRAIL_LENGTH))`
- gets existing trail for point i
- if point i is new — creates a fresh deque

#### `trails = new_trails`
- only keeps trails for points that were successfully tracked
- lost points are automatically dropped

#### Drawing the trail
- `cv2.line(pts[j-1], pts[j])` — connects consecutive positions
- `cv2.circle(pts[-1])` — marks current position in red

#### `prev_points = next_points[status == 1].reshape(-1, 1, 2)`
- between refreshes, carry forward only successfully tracked points
- `.reshape(-1, 1, 2)` — restores OpenCV's expected point shape after filtering

#### Refresh condition
- every `REFRESH_EVERY` frames — scheduled refresh
- `len(prev_points) < 10` — emergency refresh if too many points lost

### What To Expect

Still background:
- no trails anywhere
- points detected but movement below threshold

Moving hand:
- green trails follow hand movement
- direction and path clearly visible
- background stays clean

Moving camera:
- trails appear everywhere
- all pointing same direction
- called global motion

### Summary

| Concept | What It Means |
| ------- | ------------- |
| Optical flow | pixel movement between two frames |
| Sparse | track specific points only |
| Dense | track every pixel, computationally heavy |
| `goodFeaturesToTrack` | finds corners reliable enough to track |
| Cornerness | uniqueness of a pixel — makes it trackable |
| `calcOpticalFlowPyrLK` | Lucas-Kanade sparse optical flow |
| `status == 1` | point tracked successfully |
| `deque(maxlen=N)` | fixed length history — auto drops oldest |
| Trail | path of connected past positions per point |
| `REFRESH_EVERY` | how often to find fresh tracking points |
| `.reshape(-1, 1, 2)` | restores OpenCV point shape after filtering |
| `.ravel()` | flattens OpenCV point shape from `(1,2)` to `(2,)` |
| `arrowedLine` | draws movement direction and magnitude |

Next up — motion detection. Instead of tracking specific points, we detect regions where significant pixel change has occurred between frames.